In [7]:
import os, sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "src" / "pipeline" / "config.py").is_file():
    if _p.parent == _p:
        raise SystemExit("please run this notebook from inside the repository")
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

FOLD = 0  # 0-4, or None = all five folds

import numpy as np
import torch
from pipeline import config
from pipeline.test import casee_eval as ce
DEV = ce.device()
FOLDS = range(5) if FOLD is None else [FOLD]


In [8]:
from pipeline.data.seqdata import load_seqtest_file

IDX = {"wang_wt_test": 0, "xiang_nodox": 1,
       "Doench-MOLM13_NB4_TF1": 2, "Doench-Mouse_EL4": 3}
EXTERNAL = ["Wang_Xu-HL60", "Wang_Xu-KBM7"]
EXT_IDX = 0   # dataset calibration applied zero-shot


def run_fold(k):
    net, res = ce.load_sim(k, DEV), ce.load_residual(k, DEV)
    out = {}
    for name, idx in IDX.items():
        arr = load_seqtest_file(config.SIM_SEQTEST / ("%s_fold%d.dat" % (name, k)))
        y, z = ce.predict(net, res, arr, DEV, idx=idx)
        out[name] = ce.rho(y, z)
    for name in EXTERNAL:
        arr = load_seqtest_file(config.SIM_SEQTEST / (name + ".dat"))
        y, z = ce.predict(net, res, arr, DEV, idx=EXT_IDX)
        out[name] = ce.rho(y, z)
    return out


scores = {}
for k in FOLDS:
    print("--- fold %d" % k)
    for name, r in run_fold(k).items():
        scores.setdefault(name, {})[k] = r
        print("%-34s fold%d  Spearman = %.4f" % (name, k, r))

print()
for name in list(IDX) + EXTERNAL:
    v = list(scores[name].values())
    if len(v) > 1:
        print("%-34s mean = %.4f  std = %.4f  (folds=%d)" % (name, np.mean(v), np.std(v), len(v)))


--- fold 0
wang_wt_test                       fold0  Spearman = 0.7970
xiang_nodox                        fold0  Spearman = 0.7390
Doench-MOLM13_NB4_TF1              fold0  Spearman = 0.6010
Doench-Mouse_EL4                   fold0  Spearman = 0.4749
Wang_Xu-HL60                       fold0  Spearman = 0.5771
Wang_Xu-KBM7                       fold0  Spearman = 0.5916



In [9]:
import pickle as pk

k = 0 if FOLD is None else FOLD
w = pk.load(open(config.ot_weights_dir(k) / "posweights.pkl", "rb"))
print("positional weights, fold %d" % k)
print("  ".join("P%02d=%.4f" % (i + 1, w[i]) for i in range(21)))

net = ce.load_sim(k, DEV)
sites = pk.load(open(config.SIM_GETDATA / "sites_data.pkl", "rb"))
BASES = "ACGT"
print()
print("per-site local energy, fold %d  (columns: %s)" % (
    k, " ".join(a + b for a in BASES for b in BASES)))
for p in range(1, 21):
    row = []
    for b1 in BASES:
        for b2 in BASES:
            a = np.asarray(sites["%d%s_%d%s" % (p, b1, p + 1, b2)], dtype=int)
            with torch.no_grad():
                e = net(torch.LongTensor(a[:, 0]).unsqueeze(0).to(DEV),
                        torch.LongTensor(a[:, 1] - 1).unsqueeze(0).to(DEV)).item()
            row.append("%8.3f" % e)
    print("P%02d %s" % (p, " ".join(row)))


positional weights, fold 0
P01=0.6242  P02=0.6240  P03=0.6241  P04=0.2488  P05=0.2313  P06=0.5601  P07=0.4658  P08=0.4433  P09=0.4714  P10=0.5966  P11=0.6062  P12=0.5107  P13=0.4314  P14=0.7238  P15=0.6314  P16=0.6560  P17=0.8918  P18=1.4694  P19=1.5060  P20=1.4941  P21=1.4021

per-site local energy, fold 0  (columns: AA AC AG AT CA CC CG CT GA GC GG GT TA TC TG TT)
P01  -18.627  -18.649  -18.378  -18.533  -18.605  -18.694  -18.595  -18.671  -18.389  -18.586  -18.334  -18.398  -18.680  -18.796  -18.502  -18.868
P02  -18.590  -18.561  -18.357  -18.512  -18.293  -18.382  -18.283  -18.359  -18.248  -18.443  -18.191  -18.254  -18.421  -18.537  -18.273  -18.609
P03  -18.539  -18.510  -18.306  -18.461  -18.340  -18.430  -18.331  -18.406  -18.292  -18.487  -18.235  -18.298  -18.548  -18.622  -18.359  -18.694
P04  -18.801  -18.744  -18.708  -18.724  -18.752  -18.787  -18.748  -18.778  -18.724  -18.802  -18.701  -18.726  -18.784  -18.830  -18.708  -18.842
P05  -18.824  -18.814  -18.738  -18.795